# 04 — Phase 4: Metrics & Statistical Rigor

Loads all saved results and computes the full set of paper metrics.

| Metric | Description |
|---|---|
| Conflict resolution layer | mean ± std of phase_transition_layer, per category |
| Arbitration head delta | ranked table of all 144 heads |
| Ablation flip rate | % of conflict prompts flipped per top candidate |
| Patch success rate | % of prompts where losing-circuit patch flips answer |
| Cross-category Jaccard | top-10 head set overlap across A/B/C |
| t-test | Welch's t-test: conflict vs unambiguous activation magnitude |

**Output:** `data/results/phase4/paper_metrics.md` and `paper_metrics.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

from circuit_conflict.metrics import (
    phase_transition_stats,
    rank_heads_by_delta,
    cross_category_jaccard,
    arbitration_delta_score,
    t_test_conflict_vs_unamb,
    compile_paper_metrics,
)

P1 = Path('../data/results/phase1')
P2 = Path('../data/results/phase2')
P3 = Path('../data/results/phase3')
P4 = Path('../data/results/phase4')
P4.mkdir(parents=True, exist_ok=True)

## 1. Conflict resolution layer (Phase 2)

In [ ]:
df_p2 = pd.read_csv(P2 / 'phase2_summary.csv')

transition_stats = {}
for cat in ['A', 'B', 'C']:
    transitions = df_p2[df_p2.category == cat]['phase_transition_layer'].tolist()
    st = phase_transition_stats(transitions)
    transition_stats[cat] = st

# Overall
transition_stats['All'] = phase_transition_stats(df_p2['phase_transition_layer'].tolist())

df_ts = pd.DataFrame(transition_stats).T
df_ts.to_csv(P4 / 'transition_stats.csv')
print('Conflict Resolution Layer Statistics')
print(df_ts[['mean', 'std', 'median', 'detection_rate', 'n_detected', 'n_total']].round(3))

## 2. Arbitration head delta — full ranked table

In [ ]:
delta = np.load(P3 / 'activation_delta.npy')  # (12, 12)
ranked = rank_heads_by_delta(delta)
df_ranked = pd.DataFrame(ranked, columns=['layer', 'head', 'delta_score'])
df_ranked['rank'] = range(1, len(df_ranked) + 1)
df_ranked.to_csv(P4 / 'ranked_heads_full.csv', index=False)
print('Top 15 arbitration head candidates:')
print(df_ranked.head(15).to_string(index=False))

## 3. Ablation flip rate

In [ ]:
df_abl = pd.read_csv(P3 / 'ablation_results.csv')
df_abl = df_abl.sort_values('flip_rate', ascending=False)

print('Ablation Flip Rates:')
print(df_abl[['rank', 'layer', 'head', 'delta_score', 'flip_rate', 'n_flipped', 'n_prompts']].to_string(index=False))

top_flip = df_abl.iloc[0]
print(f'\nHighest flip rate: L{top_flip.layer:.0f}H{top_flip.head:.0f} → '
      f'{top_flip.flip_rate:.1%} ({top_flip.n_flipped:.0f}/{top_flip.n_prompts:.0f})')

## 4. Patch success rate

In [ ]:
df_patch = pd.read_csv(P3 / 'patch_results.csv')
overall_patch_rate = df_patch['answer_flipped'].mean()
print(f'Overall patch success rate: {overall_patch_rate:.1%} '
      f'({df_patch["answer_flipped"].sum()}/{len(df_patch)})')

print('\nPer category:')
cat_rates = df_patch.groupby('category')['answer_flipped'].agg(['mean', 'sum', 'count'])
cat_rates.columns = ['success_rate', 'n_flipped', 'n_total']
print(cat_rates)

patch_stats = {
    'overall_patch_success_rate': float(overall_patch_rate),
    'n_prompts': int(len(df_patch)),
    'n_flipped': int(df_patch['answer_flipped'].sum()),
    'by_category': cat_rates.to_dict(),
}
with open(P4 / 'patch_stats.json', 'w') as f:
    json.dump(patch_stats, f, indent=2)

## 5. Cross-category Jaccard similarity

In [ ]:
jaccard_df = pd.read_csv(P3 / 'jaccard_matrix.csv', index_col=0)
print('Cross-category Jaccard (top-10 arbitration heads):')
print(jaccard_df.round(3))

# Highlight: are A-B, A-C, B-C Jaccard scores > 0.3 (suggestive of shared mechanism)?
threshold = 0.3
pairs = [('A','B'), ('A','C'), ('B','C')]
print(f'\nPairs with Jaccard > {threshold} (suggestive of shared arbitration heads):')
for c1, c2 in pairs:
    if c1 in jaccard_df.index and c2 in jaccard_df.columns:
        j = jaccard_df.loc[c1, c2]
        flag = '← SHARED MECHANISM' if j > threshold else ''
        print(f'  {c1}–{c2}: {j:.3f} {flag}')

## 6. Welch's t-test: conflict vs. unambiguous activation magnitude

In [ ]:
conflict_mags = np.load(P3 / 'conflict_magnitudes.npy')  # (n_conf, 12, 12)
unamb_mags    = np.load(P3 / 'unamb_magnitudes.npy')     # (n_unamb, 12, 12)

ttest_results = []
top_ranked = rank_heads_by_delta(delta)[:10]

for (layer, head, delta_score) in top_ranked:
    conf_vals  = conflict_mags[:, layer, head]   # (n_conf,)
    unamb_vals = unamb_mags[:, layer, head]      # (n_unamb,)
    res = t_test_conflict_vs_unamb(conf_vals, unamb_vals)
    res.update({'layer': layer, 'head': head, 'delta_score': delta_score})
    ttest_results.append(res)

df_ttest = pd.DataFrame(ttest_results)
df_ttest['significant'] = df_ttest['p_value'] < 0.05
df_ttest.to_csv(P4 / 'ttest_results.csv', index=False)

print('Welch t-test results for top-10 arbitration candidates:')
print(df_ttest[['layer', 'head', 'delta_score', 't_stat', 'p_value', 'cohens_d', 'significant']].to_string(index=False))

## 7. Compile full paper metrics summary

In [ ]:
flip_results_by_head = {
    (int(row.layer), int(row.head)): row.to_dict()
    for _, row in df_abl.iterrows()
}

paper_md = compile_paper_metrics(
    transition_stats_by_cat=transition_stats,
    ranked_heads=ranked,
    flip_results_by_head=flip_results_by_head,
    jaccard_df=jaccard_df,
    top_k=5,
)

with open(P4 / 'paper_metrics.md', 'w') as f:
    f.write(paper_md)

print(paper_md)
print('\nPhase 4 complete. ✓')